# Урок 8.4. POST /api/generate: ответы с цитированием и стриминг

Сердце ассистента: по вопросу собрать контекст из документации, сгенерировать
ответ **только по нему**, сослаться на источники и сохранить всё в историю.
Плюс режим стриминга — SSE, в котором источники приезжают раньше первого токена.

🎯 **Цели урока:**

- Разобрать grounding-промпт и сквозную нумерацию источников [N].
- Живьём получить ответ с цитатой и честный отказ на вопрос не из базы.
- Понять протокол SSE: события sources → token* → done, ошибки внутри потока.
- Убедиться, что история пишется даже при стриминге (и почему — в finally).

**Что понадобится:** всё то же + запущенная Ollama с qwen2.5:3b.

## 1. Grounding-промпт и контракт номеров

Промпт генерации ([`services/rag.py`](project/app/services/rag.py)) собран из
правил, выстраданных в модуле 3:

```
Ты — ассистент разработчика по внутренней документации компании.
Отвечай на русском языке, кратко и по делу.
Используй ТОЛЬКО факты из блока «Контекст».
После каждого утверждения ставь ссылку на номер источника [1].
Если в контексте нет ответа на вопрос, честно скажи:
«В документации ответа не нашёл» — и не придумывай ничего от себя.
```

- «ТОЛЬКО факты из контекста» — grounding (урок 3.1);
- канонический текст отказа — детектируемый отказ (урок 3.4: отказ как фича);
- ссылки [N] — то, чего не может дообучение и умеет только RAG (урок 6.1).

**Контракт номеров** — деталь, на которой ломаются многие RAG-сервисы: номер
[1] в тексте ответа обязан указывать на тот же чанк, что `sources[0]` в JSON.
У нас это гарантировано конструктивно: `build_context` нумерует чанки
`enumerate(chunks, start=1)` — и `_to_source_refs` в роутере нумерует **тот же
список тем же способом**. Один источник истины — порядок списка `chunks`;
тест `test_build_context_numbers_sources` сторожит этот контракт.

In [ ]:
import json
import subprocess
import sys
import time
from pathlib import Path

import httpx

BASE = "http://127.0.0.1:8010"

server = subprocess.Popen(
    [sys.executable, "-X", "utf8", "-m", "uvicorn", "app.main:app",
     "--host", "127.0.0.1", "--port", "8010"],
    cwd="project", stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)
print("uvicorn pid:", server.pid)

# Старт грузит e5 (и cross-encoder, если включён) - терпеливо ждём /health.
with httpx.Client(timeout=10) as probe:
    for attempt in range(240):
        try:
            if probe.get(f"{BASE}/health").status_code == 200:
                print(f"поднялся за ~{attempt * 0.5:.0f} c")
                break
        except httpx.HTTPError:
            pass
        if server.poll() is not None:
            raise RuntimeError("uvicorn умер при старте - смотрите логи запуска")
        time.sleep(0.5)
    else:
        raise RuntimeError("сервис не поднялся за 2 минуты")

    ready = probe.get(f"{BASE}/ready")
    print("/ready:", ready.status_code, json.dumps(ready.json(), ensure_ascii=False))

client = httpx.Client(timeout=300)   # общий клиент для ячеек урока

In [ ]:
response = client.post(f"{BASE}/api/generate",
                       json={"query": "Что делать, если я закоммитил секрет в git?"})
body = response.json()
print(f"[{body['took_ms']} мс, модель {body['model']}]\n")
print("ОТВЕТ:", body["answer"], "\n")
for src in body["sources"]:
    print(f"  [{src['number']}] {src['source']}#{src['chunk_index']} (cos={src['score']:.3f})")

Разбор моего прогона:

- **Ответ grounded на 100%**: это точный конспект чанка «## Коммит секрета»
  со ссылкой [1] — и `sources[0]` действительно указывает на
  `10-secrets-security.md#2`. Контракт номеров работает.
- Модель сослалась только на [1], хотя в контексте пять источников — это
  нормально: нумеруются все кандидаты, а цитируются использованные.
- **7700 мс** — не пугайтесь: это первый запрос после старта сервиса, Ollama
  поднимала модель с диска в память (холодный старт из урока 5.3 собственной
  персоной). В истории ниже видно: следующие запросы — 1–3 секунды. Прогрев
  после деплоя — ровно та операция, которую стоит добавить в lifespan
  (обсудим в уроке 8.6).

In [ ]:
# Вопрос, ответа на который в документации НЕТ: ждём честный отказ
response = client.post(f"{BASE}/api/generate",
                       json={"query": "Сколько дней отпуска положено в году?"})
print("ОТВЕТ:", response.json()["answer"])

Отказ сработал — но присмотритесь: **«В документации ответа не нашёл [1]-[5]»**.
Модель выполнила сразу ОБА правила промпта: и отказалась канонической фразой,
и прицепила ссылки на источники (в которых ответа нет). Типичная 3B-буквальность.

Практический вывод из урока 3.4 подтверждается: детектировать отказ нужно
**по вхождению канонической подстроки**, а не по точному равенству строк —
иначе такой «отказ с хвостом» просочится как обычный ответ. И заметьте:
вопрос про отпуск — осмысленный, косинусы его соседей всё равно ~0.8 (шкала
плотная), так что порогом такое не отсечь — честный отказ генератора остаётся
последним рубежом.

## 2. Стриминг: SSE и его подводные камни

Ответ 3B-модели занимает секунды — пользователь не должен смотреть на пустой
экран. `stream=true` переключает эндпоинт на Server-Sent Events:

```
event: sources          ← СРАЗУ, до генерации: пользователь видит, ГДЕ ищем
data: {"sources": [...]}

event: token            ← много раз, по фрагменту ответа
data: {"text": "Деплой "}

event: done             ← финал с метаданными
data: {"took_ms": 2744, "model": "qwen2.5:3b"}
```

Три инженерных детали ([`routers/generate.py`](project/app/routers/generate.py)):

1. **Источники — первым событием.** Поиск занял миллисекунды, генерация —
   секунды; всё, что готово, отдаём немедленно (этот приём мы отработали
   в сервисе урока 3.2).
2. **Ошибка — событием внутри потока.** Статус 200 и заголовки уже уехали
   клиенту; если LLM упала на середине — единственный способ сообщить об этом
   не обрывом соединения, а внятно — событие `error` в потоке.
3. **История — в `finally`.** Клиент закрыл вкладку на середине стрима? Всё,
   что успело нагенерироваться, всё равно сохранится.

In [ ]:
import time

events: list[str] = []
tokens: list[str] = []
first_token_s = None
started = time.perf_counter()

with client.stream("POST", f"{BASE}/api/generate",
                   json={"query": "Как объявляется SEV1 и кто его координирует?",
                         "stream": True}) as response:
    print("HTTP", response.status_code, "|", response.headers["content-type"])
    for line in response.iter_lines():
        if line.startswith("event:"):
            events.append(line.split(" ", 1)[1])
            if events[-1] == "token" and first_token_s is None:
                first_token_s = time.perf_counter() - started
        elif line.startswith("data:") and events and events[-1] == "token":
            tokens.append(json.loads(line[5:])["text"])

print(f"события: sources={events.count('sources')}, token={events.count('token')}, "
      f"done={events.count('done')}, error={events.count('error')}")
print(f"первый токен через {first_token_s:.1f} c")
print("ОТВЕТ ИЗ СТРИМА:", "".join(tokens))

Контракт стрима выдержан буква в букву: ровно один `sources`, 42 `token`,
один `done`, ноль `error`. Главное число — **первый токен через 0.9 секунды**:
пользователь видит начало ответа почти сразу, тогда как полный ответ занял
~1.7 секунды (сравните с записью в истории ниже). На длинных ответах разрыв
драматичнее — стриминг превращает «сервис завис?» в «сервис печатает».

Тот же вопрос без стрима лежит в истории с теми же источниками — оба режима
проходят один и тот же RagService, отличается только доставка.

In [ ]:
# История: оба режима generate пишут в SQLite (стрим - в finally)
response = client.get(f"{BASE}/api/history?limit=5")
for item in response.json():
    print(f"[{item['id']}] {item['created_at']} | {item['question'][:55]}")
    print(f"     {item['took_ms']} мс | источники: {sorted(set(item['sources']))}")

## ⚠️ Частые ошибки

1. **Нет пустой строки между SSE-событиями.** Разделитель `\n\n` обязателен —
   без него клиент склеит всё в одно вечное событие. У нас он зашит в хелпер
   `_sse()`, единственное место форматирования.
2. **Прокси буферизует стрим.** nginx по умолчанию копит ответ — пользователь
   получает «стрим» одним куском в конце. Заголовок `X-Accel-Buffering: no`
   (плюс `Cache-Control: no-cache`) отправляется с каждым стримом.
3. **Ошибка LLM в стриме = оборванное соединение.** Клиент не отличит аварию
   от конца ответа. Событие `error` в потоке — единственный честный канал.
4. **История только на успехе.** Обрыв клиента — и вопрос пользователя исчез
   из истории. `finally` сохраняет частичный ответ: для анализа качества это
   данные, а не мусор.
5. **Высокая температура «для живости».** Grounding и творчество тянут в разные
   стороны: наш дефолт 0.2, а поднимать стоит только вместе с eval-контролем
   доли отказов и выдумок (урок 3.3).

## Мини-квиз

**1. Почему событие sources отправляется до генерации, а не вместе с done?**

<details><summary>Ответ</summary>

Поиск готов через миллисекунды, генерация — через секунды. Ранние источники
дают пользователю мгновенную обратную связь («ищем там-то») и позволяют UI
отрисовать ссылки заранее. Плюс если LLM упадёт, источники уже доставлены —
пользователь может пойти читать документацию руками.
</details>

**2. Клиент получил `event: error` посреди стрима. Какой HTTP-статус у этого ответа?**

<details><summary>Ответ</summary>

200 — статус и заголовки ушли при открытии стрима, менять их поздно. Именно
поэтому ошибки в стриминговых API передаются в теле потока, а мониторить их
по HTTP-кодам бесполезно — нужна метрика (у нас rag_llm_errors_total).
</details>

**3. Чем канонический текст отказа лучше, чем «модель как-нибудь скажет, что не знает»?**

<details><summary>Ответ</summary>

Фиксированную формулировку можно детектировать кодом: считать долю отказов
метрикой, тестировать отказные сценарии, показывать в UI специальную заглушку
с подсказкой. Свободный отказ («хм, в документах об этом мало...») неотличим
от ответа (урок 3.4 — отказ как фича, а не как извинение).
</details>

## 🎓 Итоги

- Grounding-промпт: только контекст, ссылки [N], канонический отказ. Контракт
  номеров держится на одном источнике истины — порядке списка chunks — и
  закреплён тестом.
- Живьём: точный ответ с [1] по документации и честный отказ на вопрос
  не из базы.
- SSE: sources → token* → done; ошибки — событием (статус уже ушёл); история —
  в finally, переживает обрыв клиента.
- Температура 0.2: ассистент по документации — не поэт.

**Дальше — урок 8.5:** инфраструктура и тесты — 73 теста без единой модели,
схемы как контракт API (строгий разбор, computed-поля, типизированные SSE-события),
health-пробы, метрики и Docker.

In [ ]:
client.close()
server.terminate()
server.wait(timeout=15)
print("uvicorn остановлен")